# 04 · SQL Case Studies: Plan, Query, Check

Knowing the keywords is half the job. The other half is turning a question into a correct query, and **proving** it is correct. In this notebook every problem follows the same steps:

1. **Question**: what exactly do we want to know?
2. **Plan**: which tables, which level (one row per what?), which traps
3. **Query**: written in small CTE steps
4. **Check**: compare with another method, or with a total we already trust

**Case studies**

1. The Cookie Cats A/B test metrics, computed in SQL and checked against pandas
2. A data quality audit of the Chinook database
3. Seven queries that return a wrong number **without any error**

> **Colour guide:** $\color{red}{\text{red}}$ marks a warning, a trap or a wrong result. $\color{green}{\text{green}}$ marks a passed check or a correct result.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats

con = sqlite3.connect(":memory:", uri=True)   # uri=True lets ATTACH open a file read-only
con.execute("ATTACH DATABASE 'file:../data/chinook.sqlite?mode=ro' AS chinook")

cookie_cats = pd.read_csv("../data/cookie_cats.csv")
cookie_cats.to_sql("players", con, index=False)   # True/False are stored as 1/0

def run(query):
    return pd.read_sql_query(query, con)

run("SELECT * FROM players LIMIT 3")

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,0,0
1,337,gate_30,38,1,0
2,377,gate_40,165,1,0


---
## Case 1: A/B test metrics in SQL

In the experimentation folder we analysed the Cookie Cats A/B test in pandas. In a company, the raw data would sit in a database, and the first numbers would come from SQL. Let's rebuild them.

**Question:** for each version of the game (`gate_30` vs `gate_40`), how many players, and what are day-1 and day-7 retention?

**Plan:**

- One table, one row per player (`userid`), so no joins and no fan-out. Still, check that `userid` is unique.
- Check the split between groups (sample ratio mismatch, A/B notebook part 1).
- Find extreme values before averaging.
- Retention = share of players with `retention_7 = 1`, so `AVG(retention_7)`, with `100.0 *` to avoid integer division.

### Step 1: is there exactly one row per player?

In [2]:
run('''
SELECT COUNT(*)               AS rows,
       COUNT(DISTINCT userid) AS distinct_players
FROM players
''')

,rows,distinct_players
0,90189,90189


$\color{green}{\text{Same number: one row per player, safe to count rows as players.}}$

### Step 2: the split between groups

In [3]:
split = run('''
SELECT version,
       COUNT(*)                                          AS players,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct
FROM players
GROUP BY version
''')
display(split)

# SQL gives the counts; the chi-square test itself is easier in Python
# docs: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chisquare.html
srm_test = stats.chisquare(split["players"])
print(f"sample ratio mismatch test: chi-square = {srm_test.statistic:.2f}, p = {srm_test.pvalue:.4f}")

,version,players,share_pct
0,gate_30,44700,49.56
1,gate_40,45489,50.44


sample ratio mismatch test: chi-square = 6.90, p = 0.0086


Same as in A/B part 1: 49.56% vs 50.44%, p = 0.0086, a $\color{red}{\text{sample ratio mismatch}}$ we keep as a known limitation.

> **Note:** a good split of work: **SQL** for counting, filtering and aggregating large tables close to the data, **Python** for the statistical test on the small result.

### Step 3: extreme values

In [4]:
run('''
SELECT userid, version, sum_gamerounds
FROM players
ORDER BY sum_gamerounds DESC
LIMIT 3
''')

,userid,version,sum_gamerounds
0,6390605,gate_30,49854
1,871500,gate_30,2961
2,3271615,gate_40,2640


One player has 49,854 rounds, about 17 times the next one. As in A/B part 1, we remove this player before computing averages.

### Step 4: retention per group

In [5]:
retention_sql = run('''
WITH clean AS (
    SELECT *
    FROM players
    WHERE sum_gamerounds < (SELECT MAX(sum_gamerounds) FROM players)
)
SELECT version,
       COUNT(*)                            AS players,
       ROUND(100.0 * AVG(retention_1), 2)  AS day1_retention_pct,
       ROUND(100.0 * AVG(retention_7), 2)  AS day7_retention_pct
FROM clean
GROUP BY version
''')
retention_sql

,version,players,day1_retention_pct,day7_retention_pct
0,gate_30,44699,44.82,19.02
1,gate_40,45489,44.23,18.20


### Check against pandas

In [6]:
clean = cookie_cats[cookie_cats["sum_gamerounds"] < cookie_cats["sum_gamerounds"].max()]
retention_pandas = (clean.groupby("version")[["retention_1", "retention_7"]].mean() * 100).round(2)

print(retention_pandas)
assert np.allclose(retention_sql["day1_retention_pct"], retention_pandas["retention_1"])
assert np.allclose(retention_sql["day7_retention_pct"], retention_pandas["retention_7"])
print("\nSQL and pandas give the same retention numbers.")

         retention_1  retention_7
version                          
gate_30        44.82        19.02
gate_40        44.23        18.20

SQL and pandas give the same retention numbers.


$\color{green}{\text{SQL and pandas agree}}$: day-7 retention is 19.02% for `gate_30` and 18.20% for `gate_40`, the numbers tested in A/B part 2.

### Going further: where does the difference come from?

**Question:** is the drop in day-7 retention the same for players who reached the gates and for those who didn't?

**Plan:** put players into buckets of rounds played with `CASE WHEN`, then compute retention per bucket and version.

In [7]:
run('''
WITH clean AS (
    SELECT * FROM players
    WHERE sum_gamerounds < (SELECT MAX(sum_gamerounds) FROM players)
),
bucketed AS (
    SELECT version,
           retention_7,
           CASE WHEN sum_gamerounds < 30 THEN '1) under 30 rounds'
                WHEN sum_gamerounds < 40 THEN '2) 30 to 39 rounds'
                ELSE                          '3) 40+ rounds'
           END AS rounds_bucket
    FROM clean
)
SELECT rounds_bucket,
       SUM(version = 'gate_30')                                                   AS players_gate_30,
       SUM(version = 'gate_40')                                                   AS players_gate_40,
       ROUND(100.0 * AVG(CASE WHEN version = 'gate_30' THEN retention_7 END), 2) AS day7_gate_30_pct,
       ROUND(100.0 * AVG(CASE WHEN version = 'gate_40' THEN retention_7 END), 2) AS day7_gate_40_pct
FROM bucketed
GROUP BY rounds_bucket
ORDER BY rounds_bucket
''')

,rounds_bucket,players_gate_30,players_gate_40,day7_gate_30_pct,day7_gate_40_pct
0,1) under 30 rounds,28044,28876,4.26,3.93
1,2) 30 to 39 rounds,3090,2786,15.73,15.69
2,3) 40+ rounds,13565,13827,50.28,48.51


**Two SQL tricks used here:**

- `SUM(version = 'gate_30')`: a comparison is 1 when true and 0 when false, so summing it counts the matching rows.
- `AVG(CASE WHEN version = 'gate_30' THEN retention_7 END)`: the `CASE` returns NULL for the other version, and `AVG` ignores NULLs, so this is the average for `gate_30` only. This "pivot" puts both versions side by side in one row.

**Reading this:** the biggest gap is among players with 40+ rounds (about 50.3% vs 48.5%), the players who actually met the gate in both versions.

> $\color{red}{\text{Warning:}}$ rounds played is measured **after** the gate was assigned, and the gate changes how much people play. So these buckets are not fair comparisons, the same "bad control" problem as in logistic regression (statistics notebook 11). The table is useful to **explore** where the effect shows up, but the test of the whole experiment must compare all players of each group.

---
## Case 2: a data quality audit

Before trusting any report, check the data. **Question:** is the Chinook database internally consistent?

**Plan:** one query per rule, each returning the number of **problem rows** (0 = pass). Then stack them into one audit table with `UNION ALL`.

| Rule | Why it matters |
|---|---|
| Primary keys are unique | otherwise counts and joins double rows |
| Every foreign key points to an existing row | orphans disappear in INNER JOINs |
| Invoice totals equal the sum of their lines | otherwise two "revenue" numbers disagree |
| Prices and quantities are positive | negative values hint at refunds or errors |
| Required columns are not NULL | NULLs break filters and averages (notebook 01) |

In [8]:
audit = run('''
SELECT 'Invoice ids are unique' AS rule,
       COUNT(*) - COUNT(DISTINCT InvoiceId) AS problem_rows
FROM chinook.Invoice

UNION ALL
SELECT 'Every invoice line points to an existing invoice',
       COUNT(*)
FROM chinook.InvoiceLine AS il
LEFT JOIN chinook.Invoice AS i ON i.InvoiceId = il.InvoiceId
WHERE i.InvoiceId IS NULL

UNION ALL
SELECT 'Every invoice line points to an existing track',
       COUNT(*)
FROM chinook.InvoiceLine AS il
LEFT JOIN chinook.Track AS t ON t.TrackId = il.TrackId
WHERE t.TrackId IS NULL

UNION ALL
SELECT 'Every invoice points to an existing customer',
       COUNT(*)
FROM chinook.Invoice AS i
LEFT JOIN chinook.Customer AS c ON c.CustomerId = i.CustomerId
WHERE c.CustomerId IS NULL

UNION ALL
SELECT 'Invoice total = sum of its lines',
       COUNT(*)
FROM chinook.Invoice AS i
JOIN (SELECT InvoiceId, SUM(UnitPrice * Quantity) AS lines_total
      FROM chinook.InvoiceLine
      GROUP BY InvoiceId) AS l ON l.InvoiceId = i.InvoiceId
WHERE ABS(i.Total - l.lines_total) > 0.005

UNION ALL
SELECT 'Prices and quantities are positive',
       COUNT(*)
FROM chinook.InvoiceLine
WHERE UnitPrice <= 0 OR Quantity <= 0

UNION ALL
SELECT 'Every invoice has a date and a total',
       COUNT(*)
FROM chinook.Invoice
WHERE InvoiceDate IS NULL OR Total IS NULL
''')
audit["result"] = np.where(audit["problem_rows"] == 0, "pass", "FAIL")
audit

,rule,problem_rows,result
0,Invoice ids are unique,0,pass
1,Every invoice line points to an existing invoice,0,pass
2,Every invoice line points to an existing track,0,pass
3,Every invoice points to an existing customer,0,pass
4,Invoice total = sum of its lines,0,pass
5,Prices and quantities are positive,0,pass
6,Every invoice has a date and a total,0,pass


$\color{green}{\text{Every rule passes.}}$

> **Note:** for the invoice totals we compare with a tolerance (`ABS(...) > 0.005`) instead of `=`. Prices like 0.99 are stored as decimal floating-point numbers, and adding them can give 3.9600000000000004 instead of 3.96. An exact `=` could report false problems.

`UNION ALL` stacks the results of several queries with the same columns. (`UNION` without `ALL` also removes duplicate rows, which is slower and here could even merge two rules that happen to have the same text and count.)

### Missing values: not errors, but worth knowing

In [9]:
run('''
SELECT 'Customer.Company' AS column_name, COUNT(*) AS rows, SUM(Company IS NULL) AS nulls,
       ROUND(100.0 * AVG(Company IS NULL), 1) AS null_pct
FROM chinook.Customer
UNION ALL
SELECT 'Customer.State', COUNT(*), SUM(State IS NULL), ROUND(100.0 * AVG(State IS NULL), 1)
FROM chinook.Customer
UNION ALL
SELECT 'Track.Composer', COUNT(*), SUM(Composer IS NULL), ROUND(100.0 * AVG(Composer IS NULL), 1)
FROM chinook.Track
''')

,column_name,rows,nulls,null_pct
0,Customer.Company,59,49,83.1
1,Customer.State,59,29,49.2
2,Track.Composer,3503,977,27.9


Most customers are private persons (no company), many countries have no states, and about 28% of tracks have no composer listed. These are real gaps, not mistakes, but a report "revenue by composer" would silently drop more than a quarter of the tracks.

---
## Case 3: seven queries that are wrong without any error

These are the mistakes from notebooks 01–03 plus one about dates, side by side. Each pair of queries looks almost the same. In every case the database runs the wrong query happily.

### The date trap

`InvoiceDate` is stored as **text** (`'2025-03-31 00:00:00'`), and SQLite compares text letter by letter. **Question:** how many invoices were there in March 2025?

In [10]:
run('''
SELECT (SELECT COUNT(*) FROM chinook.Invoice
        WHERE InvoiceDate BETWEEN '2025-03-01' AND '2025-03-31')            AS with_between,
       (SELECT COUNT(*) FROM chinook.Invoice
        WHERE InvoiceDate >= '2025-03-01' AND InvoiceDate < '2025-04-01')  AS with_half_open_range
''')

,with_between,with_half_open_range
0,5,7


$\color{red}{\text{BETWEEN misses the invoices of 31 March}}$: as text, `'2025-03-31 00:00:00'` comes **after** `'2025-03-31'`, so it is outside the range. $\color{green}{\text{The half-open range}}$ (`>= start AND < next start`) works for dates, timestamps and any time of day.

> **Note:** time zones cause the same kind of silent shift. These dates have no time zone, so we don't know if "31 March" is local time or UTC. In a real database, always check which one a timestamp uses before cutting it into days.

### The checklist

| # | Trap | Wrong | Right | Seen in |
|---|---|---|---|---|
| 1 | `AND` before `OR` | `a AND b OR c` | `a AND (b OR c)` | notebook 01 |
| 2 | NULL in comparisons | `age <> 30` drops unknown ages | add `OR age IS NULL` if they belong | notebook 01 |
| 3 | Integer division | `SUM(x) / COUNT(*)` = 0 | `1.0 * SUM(x) / COUNT(*)` or `AVG(x)` | notebook 01 |
| 4 | Join fan-out | summing invoice totals after joining lines (×9) | aggregate first, then join | notebook 02 |
| 5 | `NOT IN` + NULL | returns no rows at all | `NOT EXISTS` | notebook 02 |
| 6 | Default window frame | equal dates share one running total | `ROWS BETWEEN …` + unique order | notebook 03 |
| 7 | Text dates and `BETWEEN` | misses the last day | `>= start AND < next start` | this notebook |

**Habits that catch all of them:**

- Know the level of every table: "one row per what?"
- Count rows before and after each join.
- Count NULLs before filtering or averaging.
- Compare a total with an independent calculation (another table, another method, pandas).

---
## Summary

| Case | What we practised | Result |
|---|---|---|
| A/B test metrics | uniqueness check, split, outliers, retention, pivot with `CASE`, SQL vs pandas check | same numbers as the pandas analysis (19.02% vs 18.20%) |
| Data quality audit | uniqueness, orphans, reconciliation, ranges, NULLs, `UNION ALL` | every rule passes, 28% of tracks have no composer |
| Silent errors | 7 traps with a wrong and a right query each | a checklist to run before trusting a number |

This is the end of the SQL folder. The last folder, **guesstimates** (`04_guesstimates/`), switches from exact data to **estimating**: breaking a big unknown number into small pieces you can reason about, with assumptions, sanity checks and sensitivity analysis in Python.

## References and further reading

- [SQLite: date and time functions](https://www.sqlite.org/lang_datefunc.html) · [SQLite: compound SELECT (UNION)](https://www.sqlite.org/lang_select.html#compound_select_statements)
- [PostgreSQL: date/time types and time zones](https://www.postgresql.org/docs/current/datatype-datetime.html)
- [Sample ratio mismatch, Microsoft Research ExP](https://www.microsoft.com/en-us/research/articles/diagnosing-sample-ratio-mismatch-in-a-b-testing/)